# Reckless Mean-Reversion Strategy — Research Analysis

**Hypothesis:** Assets entering the daily Top-20 weekly gainers (ranked by trailing 7-day return) in crypto perpetual futures may experience **momentum exhaustion** and subsequently **mean revert**.

This notebook reproduces the full analysis pipeline:
1. Data inspection
2. Forward return analysis
3. Distribution analysis
4. Strategy interpretation
5. Walk-forward evaluation
6. Next research steps

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, Markdown

# Paths
BASE  = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "analysis" else os.getcwd()
OUT   = os.path.join(BASE, "outputs")
PLOTS = os.path.join(OUT, "research_plots")
os.makedirs(PLOTS, exist_ok=True)

# Dark theme
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "text.color":       "#c9d1d9",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "grid.color":       "#21262d",
    "grid.linestyle":   "--",
    "grid.alpha":       0.6,
    "font.family":      "sans-serif",
    "font.size":        11,
    "figure.dpi":       130,
})
ACCENT   = "#58a6ff"
NEGATIVE = "#f85149"
POSITIVE = "#3fb950"
WARN     = "#d29922"

print(f"Base dir: {BASE}")
print(f"Outputs:  {OUT}")

---
## Task 1 — Inspect Repository Data

In [ ]:
data_files = {
    "events.csv":                  "Individual top-20 gainer events with pre/post-signal daily returns",
    "forward_returns_by_day.csv":  "Aggregated forward-return stats per horizon day 1-14",
    "bucket_paths.csv":            "Forward-return paths by 7d-gain percentile quartile",
    "filter_oos_results.csv":      "22 filter combos evaluated on train/test split",
    "walk_forward_results.csv":    "Monthly walk-forward results across all filter × window combos",
    "walk_forward_summary.csv":    "Summary of which filters were selected in walk-forward",
}

for fname, desc in data_files.items():
    fpath = os.path.join(OUT, fname)
    size_kb = os.path.getsize(fpath) / 1024
    df_temp = pd.read_csv(fpath)
    print(f"  {fname:40s}  {size_kb:8.1f} KB   {df_temp.shape[0]:>6,} rows × {df_temp.shape[1]:>3} cols")
    print(f"    → {desc}")
    print()

In [ ]:
# Quick peek at each file
for fname in data_files:
    df_temp = pd.read_csv(os.path.join(OUT, fname))
    display(Markdown(f"### `{fname}` — first 3 rows"))
    display(df_temp.head(3))

---
## Task 2 — Forward Return Analysis

Using `forward_returns_by_day.csv` to visualize the post-event return profile.

In [ ]:
fwd = pd.read_csv(os.path.join(OUT, "forward_returns_by_day.csv"))
display(fwd)

### 2.1 — Median Forward Return vs Horizon

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(fwd["horizon_day"], fwd["median_return"] * 100, color=NEGATIVE, alpha=0.85, edgecolor="#21262d")
ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("Median Forward Return (%)")
ax.set_title("Median Forward Return vs Horizon Day", fontweight="bold", fontsize=14)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "median_return_vs_horizon.png"))
plt.show()

### 2.2 — Mean Forward Return vs Horizon

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = [POSITIVE if v >= 0 else NEGATIVE for v in fwd["mean_return"]]
ax.bar(fwd["horizon_day"], fwd["mean_return"] * 100, color=colors, alpha=0.85, edgecolor="#21262d")
ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("Mean Forward Return (%)")
ax.set_title("Mean Forward Return vs Horizon Day", fontweight="bold", fontsize=14)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "mean_return_vs_horizon.png"))
plt.show()

### 2.3 — Reversal Probability vs Horizon

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fwd["horizon_day"], fwd["reversal_probability"] * 100,
        marker="o", color=ACCENT, lw=2.5, markersize=7, zorder=3)
ax.axhline(50, color=WARN, ls="--", lw=1.2, label="50% (coin flip)")
ax.fill_between(fwd["horizon_day"], 50, fwd["reversal_probability"] * 100,
                alpha=0.15, color=ACCENT)
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("Reversal Probability (%)")
ax.set_title("Reversal Probability vs Horizon Day", fontweight="bold", fontsize=14)
ax.legend(facecolor="#161b22", edgecolor="#30363d")
ax.set_ylim(45, 70)
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "reversal_probability_vs_horizon.png"))
plt.show()

### 2.4 — Mean − Median (Tail-Effect Indicator)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
diff = (fwd["mean_return"] - fwd["median_return"]) * 100
ax.bar(fwd["horizon_day"], diff, color=WARN, alpha=0.85, edgecolor="#21262d")
ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("Mean − Median Return (%)")
ax.set_title("Mean − Median Return (Tail-Effect Indicator)", fontweight="bold", fontsize=14)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "mean_minus_median_vs_horizon.png"))
plt.show()

### 2.5 — Cumulative Median Return Path

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cum_med = fwd["median_return"].values
ax.plot(fwd["horizon_day"], cum_med * 100, marker="s", color=NEGATIVE, lw=2.5, markersize=7)
ax.fill_between(fwd["horizon_day"], 0, cum_med * 100, alpha=0.15, color=NEGATIVE)
ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("Median Return (%)")
ax.set_title("Median Return Path After Top-20 Gainer Event", fontweight="bold", fontsize=14)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "cumulative_median_return_path.png"))
plt.show()

---
## Task 3 — Distribution Analysis

Using the event-level forward return data to study the return distribution.

In [ ]:
events = pd.read_csv(os.path.join(OUT, "events.csv"))
print(f"Loaded {len(events):,} events")
print(f"Columns: {events.shape[1]}")
print(f"Date range: {events['date'].min()} → {events['date'].max()}")
print(f"Unique symbols: {events['symbol'].nunique()}")

### 3.1 — Histogram of 7-Day Forward Returns

In [ ]:
ret7 = events["ret_+7"].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ret7 * 100, bins=120, color=ACCENT, alpha=0.75, edgecolor="#21262d")
ax.axvline(ret7.median() * 100, color=NEGATIVE, ls="--", lw=1.5,
           label=f"Median = {ret7.median()*100:.2f}%")
ax.axvline(ret7.mean() * 100, color=POSITIVE, ls="--", lw=1.5,
           label=f"Mean = {ret7.mean()*100:.2f}%")
ax.axvline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("7-Day Forward Return (%)")
ax.set_ylabel("Count")
ax.set_title("Distribution of 7-Day Forward Returns", fontweight="bold", fontsize=14)
ax.legend(facecolor="#161b22", edgecolor="#30363d")
ax.set_xlim(-80, 120)
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "histogram_7d_forward_return.png"))
plt.show()

### 3.2 — Boxplot of Forward Returns by Horizon

In [ ]:
ret_cols = [c for c in events.columns if c.startswith("ret_+")]
ret_data = events[ret_cols].copy()
ret_data.columns = [c.replace("ret_+", "Day ") for c in ret_cols]
melted = ret_data.melt(var_name="Horizon", value_name="Return")
melted["Return"] *= 100
melted = melted.dropna()

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=melted, x="Horizon", y="Return", ax=ax,
            flierprops=dict(marker=".", markerfacecolor="#8b949e", markersize=2, alpha=0.3),
            boxprops=dict(facecolor="#21262d", edgecolor=ACCENT),
            medianprops=dict(color=NEGATIVE, lw=2),
            whiskerprops=dict(color="#8b949e"),
            capprops=dict(color="#8b949e"))
ax.axhline(0, color=WARN, ls="--", lw=1)
ax.set_xlabel("Horizon")
ax.set_ylabel("Forward Return (%)")
ax.set_title("Forward Return Distribution by Horizon", fontweight="bold", fontsize=14)
ax.set_ylim(-100, 150)
ax.grid(True, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "boxplot_forward_returns_by_horizon.png"))
plt.show()

### 3.3 — Percentile Table

In [ ]:
pct_rows = []
for c in ret_cols:
    day = c.replace("ret_+", "")
    s = events[c].dropna()
    pct_rows.append({
        "Horizon": f"Day {day}",
        "p5":  f"{s.quantile(0.05)*100:.2f}%",
        "p25": f"{s.quantile(0.25)*100:.2f}%",
        "p50 (median)": f"{s.quantile(0.50)*100:.2f}%",
        "p75": f"{s.quantile(0.75)*100:.2f}%",
        "p95": f"{s.quantile(0.95)*100:.2f}%",
        "Mean": f"{s.mean()*100:.2f}%",
        "N": len(s),
    })
pct_df = pd.DataFrame(pct_rows)
display(pct_df)
pct_df.to_csv(os.path.join(PLOTS, "percentile_table.csv"), index=False)

**Observation:** The p95 at Day 14 is +52.5% while the median is −5.3%. A small number of **explosive winners** drag the mean positive while the **majority of events mean-revert**. This is the core challenge for a short strategy — fat-tail winners can obliterate gains.

---
## Task 4 — Strategy Interpretation

### Does the dataset show evidence of mean reversion?

**Yes.** The median forward return is **persistently negative** at all horizons (−0.86% at day 1, −5.28% at day 14). Over **56–62%** of events produce negative returns post-signal. The "typical" top-gainer event mean-reverts.

### Why might median returns be negative while mean returns are positive?

The distribution is **heavily right-skewed**. A small subset (~5–10%) of events experience continued parabolic momentum. At day 14, the p95 reaches **+52.5%** while the p50 is **−5.3%**. These fat-tail winners pull the mean upward while the majority of events decline.

### What risks would a short strategy face?

1. **Explosive right-tail risk** — a single winner can erase dozens of winning trades
2. **Funding rate costs** — highly-shorted assets command high short funding rates in crypto perps
3. **Liquidity gaps** — moonshot coins often have thin order books
4. **Correlated drawdowns** — during market-wide rallies, top-gainer shorts can all rip simultaneously
5. **Selection bias** — in-sample filter performance may not persist OOS

### Is the base effect strong enough to justify further filtering?

**Yes.** The base reversal rate (~60%) and −5.3% median 14d return are economically meaningful. The bucket analysis shows that the top quartile of gainers reverts hardest (−10.4% median at day 14), justifying tighter cutoffs and additional filters.

---
## Task 5 — Walk-Forward Evaluation

In [ ]:
wf  = pd.read_csv(os.path.join(OUT, "walk_forward_results.csv"))
wfs = pd.read_csv(os.path.join(OUT, "walk_forward_summary.csv"))

print(f"Walk-forward results: {len(wf)} rows")
print(f"Unique test months: {wf['test_month'].nunique()}")
print(f"Unique filters tested: {wf['filter_name'].nunique()}")
print()
display(Markdown("### Walk-Forward Summary"))
display(wfs)

In [ ]:
# Keep only the selected filter per window
sel = wf[wf["selected"] == True].copy()
sel["test_month_dt"] = pd.to_datetime(sel["test_month"])
sel = sel.sort_values("test_month_dt")

print(f"Selected rows: {len(sel)} windows")
print(f"Filters selected: {sel['filter_name'].value_counts().to_dict()}")
display(sel[["test_month", "filter_name", "test_count", "test_reversal_rate",
             "test_forward_14d_median", "test_forward_14d_mean"]].reset_index(drop=True))

### 5.1 — Equity Curve (Cumulative OOS Median Return)

In [ ]:
sel["cum_median"] = sel["test_forward_14d_median"].cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sel["test_month_dt"], sel["cum_median"] * 100,
        marker="o", color=ACCENT, lw=2, markersize=5, zorder=3)
ax.fill_between(sel["test_month_dt"], 0, sel["cum_median"] * 100,
                alpha=0.12, color=ACCENT)
ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("OOS Month")
ax.set_ylabel("Cumulative Median 14d Return (%)")
ax.set_title("Walk-Forward Equity Curve (Cumulative OOS Median Return)",
             fontweight="bold", fontsize=14)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.grid(True)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "walk_forward_equity_curve.png"))
plt.show()

### 5.2 — Rolling OOS Performance by Window

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Median 14d return per window
c_med = [NEGATIVE if v < 0 else POSITIVE for v in sel["test_forward_14d_median"]]
axes[0].bar(sel["test_month_dt"], sel["test_forward_14d_median"] * 100,
            width=25, color=c_med, alpha=0.85, edgecolor="#21262d")
axes[0].axhline(0, color="#8b949e", lw=0.8)
axes[0].set_ylabel("OOS Median 14d Return (%)")
axes[0].set_title("Walk-Forward: OOS Performance by Window", fontweight="bold", fontsize=14)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
axes[0].grid(True)

# Reversal rate per window
axes[1].bar(sel["test_month_dt"], sel["test_reversal_rate"] * 100,
            width=25, color=ACCENT, alpha=0.75, edgecolor="#21262d")
axes[1].axhline(50, color=WARN, ls="--", lw=1.2)
axes[1].set_ylabel("OOS Reversal Rate (%)")
axes[1].set_xlabel("OOS Month")
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
axes[1].grid(True)

fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "walk_forward_rolling_performance.png"))
plt.show()

### 5.3 — OOS Summary Metrics

In [ ]:
oos_summary_rows = []
for _, r in wfs.iterrows():
    oos_summary_rows.append({
        "Filter": r["filter_name"],
        "Times Selected": int(r["selected_count"]),
        "Avg OOS Reversal": f"{r['avg_test_reversal_rate']*100:.1f}%",
        "Median OOS 14d Ret": f"{r['median_test_forward_14d_median']*100:.2f}%",
        "Avg OOS 14d Mean": f"{r['avg_test_forward_14d_mean']*100:.2f}%",
        "Total OOS Events": int(r["total_test_count"]),
    })
oos_df = pd.DataFrame(oos_summary_rows)
display(oos_df)
oos_df.to_csv(os.path.join(PLOTS, "walk_forward_oos_summary.csv"), index=False)

print(f"\n{'─'*50}")
print(f"  OOS Windows:         {len(sel)}")
print(f"  Win % (median < 0):  {(sel['test_forward_14d_median'] < 0).mean()*100:.1f}%")
print(f"  Avg Reversal Rate:   {sel['test_reversal_rate'].mean()*100:.1f}%")
print(f"  Avg OOS Median 14d:  {sel['test_forward_14d_median'].mean()*100:.2f}%")
print(f"  Avg OOS Mean 14d:    {sel['test_forward_14d_mean'].mean()*100:.2f}%")

### Stability Assessment

- **`trade_count_top_half`** dominates selection (13/24 windows) — strong persistence signal
- OOS reversal rates in the 67–71% range are encouraging
- The equity curve shows steady negative drift in OOS median returns
- **Caveat**: OOS mean returns are near zero — tail losers erode profitability on a mean basis
- **Position sizing and stop-losses are critical**

---
## Bonus — Bucket Path Analysis

In [ ]:
bp = pd.read_csv(os.path.join(OUT, "bucket_paths.csv"))

fig, ax = plt.subplots(figsize=(10, 6))
palette = {
    "25th percentile and below": POSITIVE,
    "25th percentile - 50th percentile": ACCENT,
    "50th percentile - 75th percentile": WARN,
    "75th percentile & up": NEGATIVE,
}

for bucket, grp in bp.groupby("bucket"):
    grp = grp.sort_values("horizon_day")
    ax.plot(grp["horizon_day"], grp["median_return"] * 100,
            marker="o", lw=2, markersize=5,
            color=palette.get(bucket, "#8b949e"),
            label=bucket)

ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_xlabel("Horizon (days)")
ax.set_ylabel("Median Forward Return (%)")
ax.set_title("Median Return Path by 7d-Gain Percentile Bucket", fontweight="bold", fontsize=14)
ax.legend(fontsize=9, facecolor="#161b22", edgecolor="#30363d", loc="lower left")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.grid(True)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "bucket_median_return_paths.png"))
plt.show()

---
## Bonus — Filter OOS Comparison

In [ ]:
filt = pd.read_csv(os.path.join(OUT, "filter_oos_results.csv"))

fig, ax = plt.subplots(figsize=(14, 7))
filt_sorted = filt.sort_values("test_reversal_rate", ascending=True)

y_pos = range(len(filt_sorted))
ax.barh(y_pos, filt_sorted["test_reversal_rate"] * 100,
        color=ACCENT, alpha=0.8, edgecolor="#21262d")
ax.axvline(50, color=WARN, ls="--", lw=1.2, label="50% baseline")
ax.set_yticks(y_pos)
ax.set_yticklabels(filt_sorted["filter_name"], fontsize=8)
ax.set_xlabel("OOS Reversal Rate (%)")
ax.set_title("Filter Performance: OOS Reversal Rate", fontweight="bold", fontsize=14)
ax.legend(facecolor="#161b22", edgecolor="#30363d")
ax.grid(True, axis="x")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS, "filter_oos_reversal_rates.png"))
plt.show()

---
## Task 6 — Suggested Next Research Steps

1. **Tighter rank cutoffs** — test Top-5 vs Top-10 vs Top-20; bucket analysis shows top quartile reverts hardest
2. **Liquidity filters** — minimum quote volume / trade count to exclude illiquid names
3. **Volatility filters** — realized vol regime conditioning; strategy likely better in non-bull markets
4. **Repeated-signal cooldown** — minimum gap between entries for the same symbol
5. **Market regime conditioning** — BTC regime overlay to enable/disable signals
6. **Dynamic position sizing** — Kelly criterion or inverse-vol weighting
7. **Stop-loss analysis** — fixed and trailing stops to truncate right-tail blowups
8. **Transaction cost modeling** — taker fees, funding rates, slippage
9. **Correlation analysis** — how correlated are simultaneous signals
10. **Ensemble filters** — combine top filters (e.g., `trade_count_top_half & ret_7d_top_quartile`)

In [ ]:
print(f"All plots saved to: {PLOTS}")
print(f"Files: {os.listdir(PLOTS)}")